In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Diagonalizacja kwantowa oparta na próbkowaniu hamiltonianów chemicznych

*Szacowane zużycie zasobów: poniżej jednej minuty na procesorze Heron r2 (UWAGA: To jest jedynie szacunek. Twój czas wykonania może się różnić.)*
## Tło
W tym samouczku pokazujemy, jak poddać przetwarzaniu końcowemu zaszumione próbki kwantowe w celu znalezienia przybliżenia stanu podstawowego cząsteczki azotu $\text{N}_2$ przy równowagowej długości wiązania — z użyciem [algorytmu diagonalizacji kwantowej opartej na próbkowaniu (SQD)](https://arxiv.org/abs/2405.05068) — dla próbek pobranych z 59-qubitowego Circuit kwantowego (52 qubity systemowe + 7 qubitów ancilla). Implementacja oparta na Qiskit jest dostępna w [dodatkach SQD do Qiskit](https://github.com/Qiskit/qiskit-addon-sqd); więcej szczegółów można znaleźć w odpowiedniej [dokumentacji](/guides/qiskit-addons-sqd) oraz w [prostym przykładzie](/guides/qiskit-addons-sqd-get-started) na początek.

SQD to technika wyznaczania wartości własnych i wektorów własnych operatorów kwantowych, takich jak Hamiltonian układu kwantowego, przy użyciu obliczeń kwantowych i rozproszonych obliczeń klasycznych łącznie. Klasyczne obliczenia rozproszone służą do przetwarzania próbek uzyskanych z procesora kwantowego oraz do rzutowania i diagonalizacji docelowego Hamiltonianu w podprzestrzeni rozpinanej przez te próbki. Dzięki temu SQD jest odporny na próbki zaburzone przez szum kwantowy i radzi sobie z dużymi Hamiltonianami — takimi jak Hamiltoniany chemiczne zawierające miliony wyrazów oddziaływania — które wykraczają poza zasięg jakichkolwiek metod dokładnej diagonalizacji. Przepływ pracy oparty na SQD składa się z następujących kroków:

1. Wybierz ansatz Circuit i zastosuj go na komputerze kwantowym do stanu referencyjnego (w tym przypadku stanu [Hartree-Focka](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)).
2. Pobierz ciągi bitów z wynikowego stanu kwantowego.
3. Uruchom procedurę *samospójnego odtwarzania konfiguracji* na ciągach bitów, aby uzyskać przybliżenie stanu podstawowego.

Wiadomo, że SQD działa dobrze, gdy docelowy stan własny jest rzadki: funkcja falowa jest oparta na zbiorze stanów bazowych $\mathcal{S} = {|x\rangle }$, którego rozmiar nie rośnie wykładniczo wraz z rozmiarem problemu.

### Chemia kwantowa
Właściwości cząsteczek są w dużej mierze zdeterminowane przez strukturę elektronów w ich wnętrzu. Jako cząstki fermionowe elektrony można opisać za pomocą formalizmu matematycznego zwanego drugą kwantyzacją. Idea polega na tym, że istnieje pewna liczba *orbitali*, z których każdy może być albo pusty, albo zajęty przez fermion. Układ $N$ orbitali opisuje zbiór fermionowych operatorów anihilacji ${\hat{a}_p}_{p=1}^N$ spełniających fermionowe relacje antykomutacji:

$$
\begin{align*}
\hat{a}_p \hat{a}_q + \hat{a}_q \hat{a}_p &= 0, \\
\hat{a}_p \hat{a}_q^\dagger + \hat{a}_q^\dagger \hat{a}_p &= \delta_{pq}.
\end{align*}
$$

Sprzężony $\hat{a}_p^\dagger$ nosi nazwę operatora kreacji.

Jak dotąd nasze rozważania nie uwzględniają spinu, który jest fundamentalną właściwością fermionów. Uwzględniając spin, orbitale tworzą pary zwane *orbitalami przestrzennymi*. Każdy orbital przestrzenny składa się z dwóch *orbitali spinowych*: jednego oznaczonego jako „spin-$\alpha$" i jednego oznaczonego jako „spin-$\beta$". Piszemy więc $\hat{a}_{p\sigma}$ dla operatora anihilacji związanego z orbitalem spinowym o spinie $\sigma$ ($\sigma \in {\alpha, \beta}$) w orbitaly przestrzennym $p$. Jeżeli przez $N$ oznaczymy liczbę orbitali przestrzennych, to łączna liczba orbitali spinowych wynosi $2N$. Przestrzeń Hilberta tego układu jest rozpięta przez $2^{2N}$ ortonormalnych wektorów bazowych oznaczonych dwuczęściowymi ciągami bitów $\lvert z \rangle = \lvert z_\beta z_\alpha \rangle = \lvert z_{\beta, N} \cdots z_{\beta, 1} z_{\alpha, N} \cdots z_{\alpha, 1} \rangle$.

Hamiltonian układu molekularnego można zapisać jako

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

gdzie $h_{pr}$ i $h_{prqs}$ to liczby zespolone zwane całkami molekularnymi, które można obliczyć na podstawie specyfikacji cząsteczki przy użyciu programu komputerowego. W tym samouczku wyznaczamy całki za pomocą pakietu oprogramowania [PySCF](https://pyscf.org/).

Szczegółowe informacje na temat wyprowadzenia Hamiltonianu molekularnego można znaleźć w podręczniku chemii kwantowej (na przykład *Modern Quantum Chemistry* Szabo i Ostlunda). Ogólne omówienie sposobu, w jaki problemy chemii kwantowej są odwzorowywane na komputery kwantowe, zawiera wykład [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE&t=900) z Qiskit Global Summer School 2024.

### Ansatz lokalnego unitarnego klastra Jastrow (LUCJ)
SQD wymaga Circuit ansatz do pobierania próbek. W tym samouczku użyjemy ansatzu [lokalnego unitarnego klastra Jastrow (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k) ze względu na jego połączenie fizycznej motywacji i przyjazności dla sprzętu.

Ansatz LUCJ jest wyspecjalizowaną postacią ogólnego ansatzu unitarnego klastra Jastrow (UCJ), który ma postać

$$
  \lvert \Psi \rangle = \prod_{\mu=1}^L e^{\hat{K}_\mu} e^{i \hat{J}_\mu} e^{-\hat{K}_\mu} | \Phi_0 \rangle
$$

gdzie $\lvert \Phi_0 \rangle$ jest stanem referencyjnym, często przyjmowanym jako stan Hartree-Focka, a $\hat{K}_\mu$ i $\hat{J}_\mu$ mają postać

$$
\hat{K}_\mu = \sum_{pq, \sigma} K_{pq}^\mu \, \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{q \sigma}
\;,\;
\hat{J}_\mu = \sum_{pq, \sigma\tau} J_{pq,\sigma\tau}^\mu \, \hat{n}_{p \sigma} \hat{n}_{q \tau}
\;,
$$

gdzie zdefiniowaliśmy operator liczby cząstek

$$
\hat{n}_{p \sigma} = \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{p \sigma}.
$$

Operator $e^{\hat{K}_\mu}$ jest rotacją orbitalną, którą można zaimplementować przy użyciu znanych algorytmów w liniowej głębokości i przy liniowej łączności.
Implementacja członu $e^{i \mathcal{J}_k}$ ansatzu UCJ wymaga łączności każdy-z-każdym lub użycia sieci fermionic swap, co jest trudne do zrealizowania na zaszumionych, przed-odpornych na błędy procesorach kwantowych o ograniczonej łączności. Idea *lokalnego* ansatzu UCJ polega na nałożeniu ograniczeń rzadkości na macierze $\mathbf{J}^{\alpha\alpha}$ i $\mathbf{J}^{\alpha\beta}$, które pozwalają na ich implementację w stałej głębokości na topologiach qubitów o ograniczonej łączności. Ograniczenia są określone przez listę indeksów wskazujących, które wpisy macierzy w górnym trójkącie mogą być niezerowe (ponieważ macierze są symetryczne, wystarczy podać tylko górny trójkąt). Indeksy te można interpretować jako pary orbitali, którym wolno oddziaływać.

Rozważmy jako przykład topologię qubitów w postaci sieci kwadratowej. Możemy umieścić orbitale $\alpha$ i $\beta$ w równoległych liniach na siatce, a połączenia między tymi liniami tworzą „szczeble" drabiny, jak pokazano poniżej:

![Diagram odwzorowania qubitów dla ansatzu LUCJ na siatce kwadratowej](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/baad4e53-5bfd-4cb4-8027-2ec26d27ecdd.avif)

Przy takim układzie orbitale o tym samym spinie są połączone topologią liniową, a orbitale o różnych spinach są połączone, gdy należą do tego samego orbitalu przestrzennego. Daje to następujące ograniczenia indeksów na macierze $\mathbf{J}$:

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, \ldots, N-1}
\end{align*}
$$

Innymi słowy, jeżeli macierze $\mathbf{J}$ są niezerowe tylko w określonych indeksach górnego trójkąta, to człon $e^{i \mathcal{J}_k}$ można zaimplementować na topologii kwadratowej bez użycia jakichkolwiek bramek swap, w stałej głębokości. Oczywiście nałożenie takich ograniczeń na ansatz zmniejsza jego wyrazistość, przez co może być wymagana większa liczba powtórzeń ansatzu.

Sprzęt IBM&reg; posiada topologię qubitów w postaci sieci heavy-hex; w tym przypadku możemy przyjąć wzorzec „zygzak", przedstawiony poniżej. W tym wzorcu orbitale o tym samym spinie są odwzorowywane na qubitach z topologią liniową (czerwone i niebieskie kółka), a połączenie między orbitalami różnych spinów pojawia się co 4. orbital przestrzenny, przy czym połączenie jest realizowane przez qubit ancilla (fioletowe kółka). W tym przypadku ograniczenia indeksów wynoszą:

$$
\begin{align*}
\mathbf{J}^{\alpha\alpha} &: \set{(p, p+1) \; , \; p = 0, \ldots, N-2} \\
\mathbf{J}^{\alpha\beta} &: \set{(p, p) \;, \; p = 0, 4, 8, \ldots (p \leq N-1)}
\end{align*}
$$

![Diagram odwzorowania qubitów dla ansatzu LUCJ na siatce heavy-hex](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### Samospójne odtwarzanie konfiguracji
Procedura samospójnego odtwarzania konfiguracji jest zaprojektowana tak, aby wydobyć jak najwięcej sygnału z zaszumionych próbek kwantowych. Ponieważ Hamiltonian molekularny zachowuje liczbę cząstek i składową Z spinu, sensowne jest wybranie Circuit ansatz, który również zachowuje te symetrie. Kiedy zastosujemy go do stanu Hartree-Focka, wynikowy stan będzie miał ustaloną liczbę cząstek i składową Z spinu w warunkach bezszumowych. Dlatego obie połówki (spin-$\alpha$ i spin-$\beta$) każdego ciągu bitów próbkowanego z tego stanu powinny mieć taką samą [wagę Hamminga](https://en.wikipedia.org/wiki/Hamming_weight) jak stan Hartree-Focka. Ze względu na obecność szumu w obecnych procesorach kwantowych niektóre zmierzone ciągi bitów naruszają tę właściwość. Prosta postać postelekcji odrzucałaby te ciągi bitów, ale jest to marnotrawstwo, ponieważ mogą one nadal zawierać pewien sygnał. Procedura samospójnego odtwarzania próbuje odzyskać część tego sygnału w przetwarzaniu końcowym. Procedura jest iteracyjna i wymaga jako danych wejściowych oszacowania średnich obsadzeń każdego orbitalu w stanie podstawowym, które jest najpierw obliczane na podstawie surowych próbek. Procedura jest uruchamiana w pętli, a każda iteracja składa się z następujących kroków:

1. Dla każdego ciągu bitów naruszającego określone symetrie odwróć jego bity za pomocą probabilistycznej procedury zaprojektowanej tak, aby przybliżyć ciąg bitów do bieżącego oszacowania średnich obsadzeń orbitalnych, uzyskując nowy ciąg bitów.
2. Zbierz wszystkie stare i nowe ciągi bitów spełniające symetrie i pobierz z nich podzbiory o ustalonym z góry rozmiarze.
3. Dla każdego podzbioru ciągów bitów zrzutuj Hamiltonian na podprzestrzeń rozpinaną przez odpowiadające wektory bazowe (opis tych wektorów bazowych znajdziesz w [poprzedniej sekcji](#quantum-chemistry)) i oblicz klasycznym komputerem przybliżenie stanu podstawowego zrzutowanego Hamiltonianu.
4. Zaktualizuj oszacowanie średnich obsadzeń orbitalnych na podstawie przybliżenia stanu podstawowego o najniższej energii.

### Diagram przepływu pracy SQD
Przepływ pracy SQD przedstawiono na poniższym diagramie:

![Diagram przepływu pracy algorytmu SQD](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)
## Wymagania
Przed rozpoczęciem tego samouczka upewnij się, że masz zainstalowane:

- Qiskit SDK w wersji 1.0 lub nowszej z obsługą [wizualizacji](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime w wersji 0.22 lub nowszej (`pip install qiskit-ibm-runtime`)
- Dodatek SQD do Qiskit w wersji 0.11 lub nowszej (`pip install qiskit-addon-sqd`)
- ffsim w wersji 0.0.58 lub nowszej (`pip install ffsim`)
## Konfiguracja

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Krok 1: Odwzorowanie wejść klasycznych na problem kwantowy
W tym samouczku znajdziemy przybliżenie stanu podstawowego cząsteczki przy równowadze w bazie cc-pVDZ. Najpierw określamy cząsteczkę i jej właściwości.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


Przed skonstruowaniem Circuit ansatzu LUCJ najpierw przeprowadzamy obliczenie CCSD w następującej komórce kodu. [Amplitudy $t_1$ i $t_2$](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) z tego obliczenia zostaną użyte do inicjalizacji parametrów ansatzu.

In [4]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


Teraz używamy [ffsim](https://github.com/qiskit-community/ffsim) do stworzenia Circuit ansatzu. Ponieważ nasza cząsteczka ma stan Hartree-Focka z zamkniętą powłoką, używamy zbilansowanej spinowo odmiany ansatzu UCJ: [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). Przekazujemy pary oddziaływań odpowiednie dla topologii qubitów w postaci sieci heavy-hex (patrz [sekcja tła dotycząca ansatzu LUCJ](#local-unitary-cluster-jastrow-lucj-ansatz)). Ustawiamy `optimize=True` w metodzie `from_t_amplitudes`, aby włączyć „skompresowaną" podwójną faktoryzację amplitud $t_2$ (szczegóły znajdziesz w [The local unitary cluster Jastrow (LUCJ) ansatz](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) w dokumentacji ffsim).

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


Zalecamy następujące kroki, aby zoptymalizować ansatz i uczynić go kompatybilnym ze sprzętem.

- Wybierz fizyczne Qubit (`initial_layout`) z docelowego sprzętu, które są zgodne ze wzorcem „zig-zag" opisanym wcześniej. Rozmieszczenie Qubit w tym wzorcu prowadzi do wydajnego, kompatybilnego ze sprzętem Circuit z mniejszą liczbą Gate. Poniżej zamieszczono kod automatyzujący dobór wzorca „zig-zag", który używa heurystyki punktacji w celu minimalizacji błędów związanych z wybranym układem.
- Wygeneruj etapowy menedżer przebiegów (pass manager) przy użyciu funkcji [generate_preset_pass_manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) z Qiskit, podając wybrany `backend` i `initial_layout`.
- Ustaw etap `pre_init` swojego etapowego menedżera przebiegów na `ffsim.qiskit.PRE_INIT`. `ffsim.qiskit.PRE_INIT` zawiera przejścia Transpiler Qiskit, które rozkładają Gate na rotacje orbitalne, a następnie łączą je ze sobą, co skutkuje mniejszą liczbą Gate w końcowym Circuit.
- Uruchom menedżer przebiegów na swoim Circuit.
<details>
<summary>Kod do automatycznego doboru układu „zig-zag"</summary>

</details>

In [9]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

## Krok 3: Wykonanie za pomocą prymitywów Qiskit
Po zoptymalizowaniu obwodu do wykonania na sprzęcie jesteśmy gotowi uruchomić go na docelowym urządzeniu i zebrać próbki do estymacji energii stanu podstawowego. Ponieważ mamy tylko jeden obwód, użyjemy [trybu wykonania zadań](/guides/execution-modes) Qiskit Runtime i wykonamy nasz Circuit.

In [12]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](https://qiskit.github.io/qiskit-addon-sqd/how_tos/integrate_dice_solver.html) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>